# Mutability and Hashability
- It is not wrong to say that dict keys and set elements must be immutable but not complete either
    - Set elements, dict keys can only be what are called "hashables"
- Python uses **duck-typing** to define hashables i.e. no "parent class" called "Hashable"
    - an object is hashable if it implements the `__hash__()` and `__eq__()` methods
- A Hashable $\Rightarrow$ Immutable but not the other way round
    - A tuple is immutable but hashable only if each of its elements is hashable

In [1]:
my_tuple = ( 1, 2, 2 )
# The tuple is well formed
print( my_tuple )
# All elements of the tuple are hashable so the tuple is itself hashable -- good to go as a dict key
my_dict = { my_tuple : 1 }
print( my_dict )

my_tuple = ( 1, 2, [ 1, 2 ] )
# The tuple is well formed -- it can contain immutables / non-hashables even though it is itself immutable
print( my_tuple )
# However, this tuple cannot be made into a key ( or a set element ) since it is un-hashable since one of its elements is unhashable
my_dict = { my_tuple : 1 }
print( my_dict )

(1, 2, 2)
{(1, 2, 2): 1}
(1, 2, [1, 2])


TypeError: unhashable type: 'list'

# Slicing Issues
- Python slicing is weird in its default behavior
    - Either stick to the most common idioms e.g. `::-1`
    - Else explicitly index using `range()` that has well-defined behavior
    - See https://docs.python.org/3/library/stdtypes.html#range
# Silent Success 
- Usually one talks about **silent failure** in programming
    - Happens when a routine or attempt fails but without generating any error / warning
    - Not a nice thing since it hinders debugging
- Python has a **silent success** problem
    - Certain routines will quietly ignore illegal values
    - Also not a nice thing since it also hinders debugging
    - The routine that generated those illegal values may go unnoticed

In [2]:
a = list( range( 10 ) )
print( "the list:", a )
print( "the list reversed the idiomatic / Pythonic way:", a[ ::-1 ] )
print( "by the same logic, this should work but does not:", a[ len ( a ) - 1 : -1 : -1 ] )
print( "this works but omits the first element:", a[ len ( a ) - 1 : 0 : -1 ] )

# range is much more reliable
print( [ a[ i ] for i in range( len( a ) - 1, -1, -1 ) ] )

# This works for some reason
print( "works:", a[ : -len( a ) - 1 : -1 ] )

# Silent success
print( "also works for some reason ( silent success ):", a[ : -len( a ) - 400 : -1 ] )

# Loud failure
print( "this will fail:", a[ 400 ])

the list: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
the list reversed the idiomatic / Pythonic way: [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
by the same logic, this should work but does not: []
this works but omits the first element: [9, 8, 7, 6, 5, 4, 3, 2, 1]
[9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
works: [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
also works for some reason ( silent success ): [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]


IndexError: list index out of range

# Sets
- Can conceptually think of as dictionaries but without the values -- a _key-only_ store
    - Most commmonly used to remove duplicates 🤣
    - Elements of a set must be immutable
    - However, dictionaries and sets are quite distinct in terms of their API
- Can use the `set()` function to convert a list to a set -- not available for dict
- Supports `add()`, `remove()`, `pop()` (risky), `clear()`, `union()`, `update()`, `intersection()`, `intersection_update()`, `difference()`, `symmetric_difference()`, `issubset()`, `issuperset()`, `isdisjoint()`
- Supports cool-looking ( but possibly confusing ) overloading
    - Bitwise operators can be used in place of the above methods
    - `|` for `union()`, `|=` for `update()`, `&` for `intersection()`, `&=` for `intersection_update()`, `<=` for subset, etc
- (Nested) set comprehension works similarly to list comprehension, except that elements must be non-mutable and it creates a set 

In [3]:
# Set comprehension
my_set = { str( i ) for i in range( 10 ) }

# The order in which the set elements are printed depends on the internal implementation
# A long time ago, it used to depend on the hash value alone but this seems to have changed now
# The hash is seeded but at the beginning of every iPython session
# Running this again and again will not change the order ( as seed remains the same )
# However, if you restart your kernel, then you will get a different order
# However, however, sets are supposed to be unordered so this print ordering is dismissed as an implementation issue
print( my_set )

{'2', '0', '4', '1', '5', '8', '3', '6', '7', '9'}


# Functions
- An essential element of modular programming
- Helps isolate bugs, facilitate code-reuse, collaborations
- Functions operate on arguments, parameters and returned value(s)
    - Python is very permissive about argument types and even return types
    - Functions can return a single object or (a tuple of) multiple objects
    - No restriction that a function must return objects of a single type
    - A function that returns nothing returns `None`

In [4]:
# Function demo -- heterogenous return
# Type hints are merely polite suggestions
# This function is supposed to take two integers and return a string
# However, the input may cause it to return a tuple
def my_func( a : int, b : int ) -> str:
    if a < b:
        return "Less"
    elif a > b:
        return "More"
    else:
        return a + b, "="

a = int( input( "$ " ) )
b = int( input( "$ " ) )
print( my_func( a , b ) )

$  1
$  1


(2, '=')


# Functions -- Python is not C
- Python is neither pass-by-value nor pass-by-reference, it is pass-by-assignment
    - The passed argument is given a new name -- the parameter
    - If the passed argument is an immutable
        - Any change inside the function will simply create a new object
        - Upon returning, the calling function will not see a change in value
    - If the passed argumet is a mutable
        - Any change in the function will change the original object
        - Upon returning, the calling function will see a change in value

In [5]:
# Pass by assignment??
def my_func( a : int, b : list ) -> float:
    # a, b have local scope
    # These are local names valid only within the body ( scope ) of the function
    a += 1
    b.append( a )
    print( a, b )
    return None

a = 1
b = [ 1, 3 ]
print( a, b )
my_func( a, b )
print( a, b )

1 [1, 3]
2 [1, 3, 2]
1 [1, 3, 2]


# Function arguments -- a beautiful mess
- Two types of arguments -- positional and named
    - What is positional and what is named does not always depend on the function definition
    - It depends on who is calling and how -- but all positional arguments must come before any keyword arguments
    - There are a few exceptions
        - **Exception 1**: Named arguments must come after positional parameters
            - For example, when invoking `print()`, the named parameters `sep`, `end` always come at the end
        - **Exception 2**: the / _positional enforcer_
            - See https://docs.python.org/3/faq/programming.html#what-does-the-slash-in-the-parameter-list-of-a-function-mean   

In [6]:
def my_func( a, b, c, d ):
    print( a, b, c, d )

def my_func2( a, b, /, c, d ):
    print( a, b, c, d )

my_a = 'a'
my_b = "b"
my_c = "c"
my_d = 'd'

my_func( my_d, my_c, my_a, my_b )
my_func( d = my_d, c = my_c, a = my_a, b = my_b )

my_func2( my_d, my_c, my_a, my_b )
my_func2( d = my_d, c = my_c, a = my_a, b = my_b )

d c a b
a b c d
d c a b


TypeError: my_func2() got some positional-only arguments passed as keyword arguments: 'a, b'

# Functions -- type hints
- These are just polite requests, not enforced even in the slightest
- If you really want to enforce types
    - Use static _type checkers_: mypy ( native ), pyright ( Microsoft ), pyrefly ( Meta ), pytype ( Google -- now obsolete )
    - These will not be able to check for _dynamic_ type violations e.g. those introduced by input
- If you really really want to enforce types
    - Do dynamic type checking yourself, say using the in-built function `isinstance()`

In [7]:
print( type( my_func ) )
def my_func( a: int, b: float ):
    # Dynamic type checking
    if not isinstance( a, int ):
        raise Exception( "Please send an int" )
    if not isinstance( b, float ):
        raise Exception( "Please send a float" )
    
    print( a, b )
    return "success"

print( my_func( 1, 2.0 ) )
print( my_func( 1, 2 ) )

<class 'function'>
1 2.0
success


Exception: Please send a float

### Example of enforced positional parameters

In [8]:
# dict.fromkeys()
my_list = list( range( 10 ) )
print( dict.fromkeys( my_list, "str" ) )
print( dict.fromkeys( my_list, value = "str" ) )

{0: 'str', 1: 'str', 2: 'str', 3: 'str', 4: 'str', 5: 'str', 6: 'str', 7: 'str', 8: 'str', 9: 'str'}


TypeError: dict.fromkeys() takes no keyword arguments

# Misuse of Python's good nature
- Can even use reserved keywords as variables
- Splendid way to make life more complicated than what it usually is
- That is why this is the last slide of this talk :)

In [9]:
# Misuse demo
my_print = print
print = 10
print( "Hello World" )
# Even this will not work -- why?
my_print( "Hello World" )

TypeError: 'int' object is not callable